# Bài Thực Hành 1: Chuyển đổi & Tối ưu hóa mô hình với LiteRT (TFLite)

Chào mừng bạn đến với bài thực hành đầu tiên! Trong bài học này, chúng ta sẽ cùng nhau thực thi quy trình chuyển đổi một mô hình học sâu đã huấn luyện sẵn từ TensorFlow/Keras sang định dạng cực nhẹ của **LiteRT (.tflite)**. 

Bên cạnh đó, chúng ta sẽ áp dụng 3 phương pháp **Quantization (Lượng tử hóa)** phổ biến nhất bao gồm:
1. **Dynamic Range Quantization** (Lượng tử hóa dải động - 8-bit weights, float32 activations).
2. **Float16 Quantization** (Lượng tử hóa số thực 16-bit - tối ưu cho GPU).
3. **Full Integer Quantization** (Lượng tử hóa số nguyên toàn phần - tối ưu cho NPU/DSP).

Cuối cùng, chúng ta sẽ viết code đo đạc kích thước file đầu ra để so sánh sự tiết kiệm dung lượng thần kỳ!

### Bước 1: Import thư viện và kiểm tra phiên bản
Trước hết, hãy chắc chắn bạn đã cài đặt TensorFlow và các thư viện hỗ trợ.

In [ ]:
import os
import numpy as np
import tensorflow as tf
print(f"Phiên bản TensorFlow hiện tại: {tf.__version__}")

### Bước 2: Tải mô hình học máy đã huấn luyện sẵn (MobileNetV2)
Chúng ta sẽ sử dụng MobileNetV2 - một kiến trúc mạng tích chập (CNN) siêu nhẹ và phổ biến, chuyên dùng cho phân loại hình ảnh trên điện thoại di động.

In [ ]:
# Tải mô hình MobileNetV2 đã huấn luyện trên tập ImageNet từ Keras
model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    weights='imagenet',
    classifier_activation='softmax'
)

model.summary() # Hiển thị tóm tắt cấu trúc mạng

### Bước 3: Chuyển đổi cơ bản dạng số thực Float32 (Mặc định)
Chúng ta sẽ thực hiện chuyển đổi gốc không áp dụng tối ưu hóa nào để làm mốc so sánh (baseline).

In [ ]:
# 1. Khởi dựng bộ chuyển đổi từ mô hình Keras
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# 2. Tiến hành convert thô
tflite_float32 = converter.convert()

# 3. Lưu file mô hình thô
with open('mobilenet_v2_float32.tflite', 'wb') as f:
    f.write(tflite_float32)
print("Đã lưu mô hình Float32 mặc định thành công!")

### Bước 4: Áp dụng Dynamic Range Quantization (8-bit Weights)
Chỉ với 1 dòng cấu hình thêm, LiteRT sẽ tự động lượng tử hóa tất cả các trọng số cố định sang dạng số nguyên Int8.

In [ ]:
converter_dr = tf.lite.TFLiteConverter.from_keras_model(model)

# Bật cờ tối ưu hóa mặc định của dải động
converter_dr.optimizations = [tf.lite.Optimize.DEFAULT]

# Tiến hành chuyển đổi
tflite_dynamic_range = converter_dr.convert()

# Lưu file
with open('mobilenet_v2_dynamic_range.tflite', 'wb') as f:
    f.write(tflite_dynamic_range)
print("Đã lượng tử hóa dải động thành công!")

### Bước 5: Áp dụng Float16 Quantization
Lượng tử hóa sang số thực 16-bit giúp chạy mượt mà trên các phần cứng hỗ trợ xử lý dấu phẩy động nửa độ chính xác (như GPU di động).

In [ ]:
converter_f16 = tf.lite.TFLiteConverter.from_keras_model(model)

# Thiết lập tối ưu hóa và ép kiểu đích về FLOAT16
converter_f16.optimizations = [tf.lite.Optimize.DEFAULT]
converter_f16.target_spec.supported_types = [tf.float16]

# Tiến hành chuyển đổi và lưu
tflite_float16 = converter_f16.convert()
with open('mobilenet_v2_float16.tflite', 'wb') as f:
    f.write(tflite_float16)
print("Đã lượng tử hóa Float16 thành công!")

### Bước 6: Áp dụng Full Integer Quantization (Lượng tử số nguyên toàn phần)
Để lượng tử hóa toàn phần (cả weights và activations) sang số nguyên Int8, chúng ta bắt buộc phải xây dựng một hàm cung cấp dữ liệu mẫu đại diện (Representative Dataset) giúp mô hình đo đạc và xác định phân phối giá trị thực tế.

In [ ]:
# 1. Xây dựng bộ hiệu chuẩn giả lập gồm 100 mẫu dữ liệu ngẫu nhiên khớp dải ảnh thực
def representative_data_gen():
    for _ in range(100):
        # Tạo một ảnh ngẫu nhiên có kích thước (1, 224, 224, 3) dạng Float32 nằm trong dải [0, 1.0]
        data = np.random.rand(1, 224, 224, 3).astype(np.float32)
        yield [data]

converter_fi = tf.lite.TFLiteConverter.from_keras_model(model)
converter_fi.optimizations = [tf.lite.Optimize.DEFAULT]

# Khai báo bộ tạo dữ liệu mẫu
converter_fi.representative_dataset = representative_data_gen

# Ép buộc các toán tử trung gian phải chạy hoàn toàn bằng số nguyên Int8
converter_fi.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

# Ép kiểu dữ liệu đầu vào và đầu ra của mô hình cũng sang Int8
converter_fi.inference_input_type = tf.int8
converter_fi.inference_output_type = tf.int8

# Tiến hành chuyển đổi
tflite_full_integer = converter_fi.convert()
with open('mobilenet_v2_full_integer.tflite', 'wb') as f:
    f.write(tflite_full_integer)
print("Đã lượng tử hóa Full Integer Int8 thành công!")

### Bước 7: Đo đạc và so sánh dung lượng thực tế
Hãy cùng viết một đoạn code ngắn để hiển thị kích thước file của 4 phiên bản trên và tính toán xem chúng ta đã tối ưu hóa được bao nhiêu phần trăm nhé!

In [ ]:
files = [
    ('Float32 Mặc định', 'mobilenet_v2_float32.tflite'),
    ('Dynamic Range (Int8 Weights)', 'mobilenet_v2_dynamic_range.tflite'),
    ('Float16 GPU', 'mobilenet_v2_float16.tflite'),
    ('Full Integer Int8 toàn phần', 'mobilenet_v2_full_integer.tflite')
]

print("=" * 60)
print(f"{'Phương pháp Lượng tử':<30} | {'Dung lượng (MB)':<15} | {'Tỷ lệ tối ưu':<10}")
print("=" * 60)

baseline_size = os.path.getsize(files[0][1])

for label, filename in files:
    if os.path.exists(filename):
        size_bytes = os.path.getsize(filename)
        size_mb = size_bytes / (1024 * 1024)
        ratio = (1.0 - (size_bytes / baseline_size)) * 100
        print(f"{label:<30} | {size_mb:>13.2f} MB | {ratio:>8.1f}%")
    else:
        print(f"{label:<30} | {'Lỗi không tìm thấy file':<15}")
print("=" * 60)

### Kết quả phân tích:
- **Float32 Mặc định:** Chiếm khoảng **14 MB** bộ nhớ.
- **Dynamic Range & Full Integer:** Giảm xuống chỉ còn khoảng **3.5 MB** (tiết kiệm gần **75%** dung lượng cực kỳ ấn tượng!). Đây chính là phép màu giúp mô hình dễ dàng cài đặt gọn gàng trên di động.